## RAG 第 3 天

### InsureLLM 的专家问答器

基于 LangChain 1.0 实现的 RAG 流水线。

使用我们上次创建的 VectorStore（配合 Hugging Face 的 `all-MiniLM-L6-v2`）

In [ ]:
# 导入：ChatOpenAI 对话模型；Chroma 向量库；HuggingFace 嵌入；Gradio 界面

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [ ]:
# 选择低成本聊天模型，并指定昨天建好的向量库目录 vector_db

MODEL = "gpt-4.1-nano"
DB_NAME = "vector_db"
load_dotenv(override=True)

### 连接到 Chroma；使用 Hugging Face 的 all-MiniLM-L6-v2

In [ ]:
# 加载与建库时相同的嵌入模型（embedding），才能正确查询向量
# 连接已有的 Chroma 向量数据库（vector DB），不重新写入

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

### 设置 2 个关键的 LangChain 对象：retriever 与 llm

#### 关于 “temperature” 的旁注：
- 控制输出的多样性
- temperature 为 0 意味着输出应当是可预测的
- 更高的 temperature 会带来更多样的回答

有人把 temperature 说成类似「创造力」，但那并不完全准确
- 它实际控制的是推理过程中选择哪些 token
- temperature=0 表示：始终选择概率最高的 token
- temperature=1 通常表示：概率为 10% 的 token 大约有 10% 的机会被选中

注意：temperature 为 0 并不意味着输出总是可复现的。你还需要设置随机种子。我们将在第 6–8 周这样做。（即便如此，也并不总是可复现。）

注意 2：如果你想要创造力，请使用 System Prompt！

In [ ]:
# retriever（检索器）：把问题变成向量，在库里找最相似的 chunk
# llm：回答问题的大语言模型；temperature=0 让输出更稳定

retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL)

### 这些 LangChain 对象实现了 `invoke()` 方法

In [ ]:
# 只测检索：问 "Who is Avery?"，看看返回哪些相关文档块

retriever.invoke("Who is Avery?")

In [ ]:
# 只测 LLM（无 RAG）：不给知识库上下文，模型可能答错或幻觉

llm.invoke("Who is Avery?")

## 是时候把它们组合起来了！

In [ ]:
# 系统提示词模板：把检索到的 context 填进 {context}
# 这就是 RAG——用检索结果增强生成（Retrieval Augmented Generation）

SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [ ]:
# 完整 RAG 流程：retrieve → 拼 context → 填进 system prompt → 调用 LLM 生成答案

def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [ ]:
# 端到端试问：名字有拼写差异时，向量检索仍可能命中相关员工文档

answer_question("Who is Averi Lancaster?", [])

## 接下来还可能发生什么？😂

In [ ]:
# 启动 Gradio 聊天界面，把 answer_question 接到前端

gr.ChatInterface(answer_question).launch()

## 承认吧——你以为 RAG 会比这复杂得多！！